# State of Data Brazil (2023, 2024, 2025-2026) — Tratamento e Consolidação com PySpark

Desafio de pós-graduação em Análise de Dados: tratar 3 bases da pesquisa **State of Data Brazil**
(Data Hackers), de 3 edições diferentes, e consolidá-las em uma única base final usando PySpark.

**Como este notebook está organizado:**

1. Setup do PySpark no Colab
2. Upload dos 3 arquivos
3. Exploração inicial (profiling) de cada base, separadamente
4. Mapeamento das colunas equivalentes entre os 3 anos
5. Decisão e justificativa da estratégia de consolidação
6. Tratamento (limpeza + padronização) de cada base
7. Consolidação final (union) e validação
8. Exportação da base final (Parquet + CSV) e relatório

Todo o código abaixo foi validado rodando contra os arquivos reais antes de ser
organizado neste notebook.

## 1. Setup do PySpark

In [ ]:
# No Google Colab o PySpark nao vem pre-instalado.
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("state_of_data_br_consolidacao")
    .master("local[*]")
    # As bases tem ~400 colunas cada; evitar encadear muitas transformacoes
    # coluna a coluna evita estourar a stack da JVM durante a analise do
    # plano (mais detalhes no Passo 6). Este ajuste e so uma seguranca extra.
    .config("spark.driver.extraJavaOptions", "-Xss64m")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark

## 2. Upload dos arquivos

Rode a célula abaixo e selecione **4 arquivos** quando solicitado: os 3 CSVs da pesquisa
(State of Data BR 2023, 2024 e 2025-2026) **e a planilha `crosswalk_manual_recebido.csv`**
(o mapeamento entre colunas dos 3 anos, revisado manualmente — ver seção 4). Se preferir usar o
Google Drive, comente o bloco de `files.upload()` e ajuste `DATA_DIR` para a pasta do seu Drive.

In [ ]:
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

try:
    from google.colab import files
    print("Selecione os 4 arquivos: State of Data BR 2023, 2024, 2025-2026 e crosswalk_manual_recebido.csv")
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, os.path.join(DATA_DIR, fname))
except ImportError:
    print("google.colab nao disponivel (rodando fora do Colab) - "
          "coloque os 4 arquivos manualmente em", DATA_DIR)

print(os.listdir(DATA_DIR))

In [ ]:
# Os nomes de arquivo do Kaggle podem variar um pouco dependendo de como voce
# baixou/renomeou cada CSV. Em vez de depender de um nome exato (o que ja causou
# erro de "arquivo nao encontrado" em testes anteriores), detectamos automaticamente
# o arquivo de cada ano procurando por CSVs cujo nome contenha o ano correspondente.
import glob

def _find_file(year, patterns):
    candidates = []
    for pat in patterns:
        candidates.extend(glob.glob(os.path.join(DATA_DIR, f"*{pat}*.csv")))
    candidates = sorted(set(candidates))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise FileNotFoundError(
            f"Mais de um CSV em {DATA_DIR} parece corresponder ao ano {year}: {candidates}. "
            f"Edite a celula abaixo e defina o caminho manualmente em FILES[{year}]."
        )
    raise FileNotFoundError(
        f"Nenhum CSV encontrado para o ano {year} em {DATA_DIR} "
        f"(procurei nomes contendo {patterns}). "
        f"Arquivos disponiveis nessa pasta: {os.listdir(DATA_DIR)}. "
        f"Confira se o upload da celula anterior funcionou e, se o nome do seu "
        f"arquivo for muito diferente do esperado, defina o caminho manualmente "
        f"em FILES[{year}] na celula abaixo."
    )

# Se a deteccao automatica falhar, e so trocar a linha correspondente por, por
# exemplo: 2024: os.path.join(DATA_DIR, "nome_exato_do_seu_arquivo.csv"),
FILES = {
    2023: _find_file(2023, ["2023"]),
    2024: _find_file(2024, ["2024"]),
    2025: _find_file(2025, ["2025", "2026"]),
}
FILES

## 3. Exploração inicial (profiling) de cada base

Antes de decidir como tratar e consolidar os dados, é preciso entender o que cada base tem:
schema, volume, nulos, duplicatas e tipos.

In [ ]:
def profile(year, path):
    df = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
    n_rows = df.count()
    n_cols = len(df.columns)
    n_dup = n_rows - df.dropDuplicates().count()
    print(f"===== {year} =====")
    print(f"arquivo: {os.path.basename(path)}")
    print(f"linhas: {n_rows} | colunas: {n_cols} | duplicatas exatas: {n_dup}")
    print("primeiras 5 colunas:", df.columns[:5])
    print("amostra:")
    df.show(3, truncate=40)
    return df

df23_raw = profile(2023, FILES[2023])
df24_raw = profile(2024, FILES[2024])
df25_raw = profile(2025, FILES[2025])

In [ ]:
# Contagem de nulos por coluna (top 10 colunas mais vazias de cada base) - PySpark
#
# Nota: os cabecalhos originais de 2023 contem pontos dentro do proprio texto do
# rotulo da pergunta (ex. "...companhia."), o que faz o parser de nomes do Spark
# (F.col/df[nome]) confundir o ponto com separador de campo aninhado -- mesmo com
# o nome inteiro entre crases. Por isso renomeamos para nomes posicionais seguros
# (toDF) so para este calculo, e resolvemos o nome original de volta so no print.
def top_nulls(df, year, n=10):
    total = df.count()
    original_cols = df.columns
    safe_names = [f"c{i}" for i in range(len(original_cols))]
    safe_df = df.toDF(*safe_names)
    counts = safe_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in safe_names]).collect()[0].asDict()
    pct = sorted(
        ((original_cols[int(k[1:])], round(100 * v / total, 1)) for k, v in counts.items()),
        key=lambda x: -x[1],
    )
    print(f"--- {year}: colunas com mais nulos ---")
    for c, p in pct[:n]:
        print(f"  {c}: {p}%")

top_nulls(df23_raw, 2023)
top_nulls(df24_raw, 2024)
top_nulls(df25_raw, 2025)

**Achado importante:** os cabeçalhos de coluna de cada ano usam um formato diferente para
codificar a mesma informação (código da pergunta + rótulo):

- **2023**: string de tupla Python, ex. `"('P1_a ', 'Idade')"`
- **2024/2025**: `"1.a_idade"` (código.subcódigo seguido de `_` + rótulo; em alguns casos o
  separador é um espaço em vez de `_`)

E as 3 bases **não têm nenhuma coluna-chave em comum** (2023 tem um `id` sequencial; 2024/2025
têm um `token` de envio do formulário — não são a mesma coisa e não permitem casar uma linha de
um ano com uma linha de outro).

## 4. Mapeamento das colunas equivalentes entre os 3 anos

Como os nomes de coluna sao formatados de forma diferente em cada ano, cada cabecalho tem um
**codigo de pergunta** embutido (ex. `P1_a` em 2023, `1.a` em 2024/2025) que, em teoria, identifica
a mesma pergunta nos 3 anos. Na pratica isso nem sempre e verdade: o questionario foi reestruturado
entre edicoes, e o mesmo codigo por vezes passou a representar uma pergunta ou opcao diferente.

### 4.1 Metodologia: crosswalk revisado manualmente

A primeira versao deste pipeline tentou resolver esses casos de forma automatica, usando
similaridade de texto entre os rotulos das perguntas/opcoes para detectar e corrigir
divergencias entre anos. Essa abordagem funcionou para a maioria dos casos, mas uma auditoria
manual dos dados encontrou repetidos casos reais em que a heuristica errava — incluindo colunas
onde conteudo de anos diferentes e semanticamente incompativel acabava casado sob o mesmo nome de
coluna. Dado o custo de um erro desses (mistura silenciosa de respostas de perguntas diferentes),
a correspondencia final entre os ~460 codigos de pergunta das 3 edicoes foi **revisada
manualmente, um a um**, numa planilha (`crosswalk_manual_recebido.csv`, enviada junto com os 3
CSVs na secao 2):

1. A planilha parte do casamento **literal** por codigo entre os 3 anos (sem nenhuma correcao
   automatica) — codigo, rotulo e coluna original de cada ano, lado a lado.
2. Cada linha foi lida e comparada manualmente. Quando os rotulos batiam entre os anos presentes,
   a linha ficou como estava. Quando nao batiam, o conteudo do ano divergente foi movido
   manualmente para a linha correta (localizada pelo texto do rotulo) dentro da mesma secao,
   cobrindo os casos de questionario renumerado/reestruturado entre edicoes.
3. Quando um codigo foi reaproveitado para uma pergunta ou lista de opcoes **genuinamente
   diferente** entre edicoes (sem correspondente real nos outros anos), esse ano foi separado em
   uma coluna propria (coluna `ok_ou_ajustar` da planilha marcada como `SEPARAR:20AA`), em vez de
   arriscar juntar conteudo incompativel.
4. A cobertura e validada **de forma programatica, contra os arquivos realmente enviados nesta
   execucao** (celula abaixo): toda coluna original dos 3 CSVs precisa aparecer em exatamente uma
   linha da planilha — nem faltando, nem duplicada. Se algum CSV enviado for diferente do que a
   planilha espera, essa celula falha com um erro explicando o que nao bateu, em vez de gerar uma
   base final incorreta silenciosamente.

In [ ]:
# Localiza e carrega a planilha de crosswalk (enviada na secao 2 junto com os CSVs).
import glob
import pandas as pd

_crosswalk_candidates = glob.glob(os.path.join(DATA_DIR, "*crosswalk*.csv"))
if len(_crosswalk_candidates) != 1:
    raise FileNotFoundError(
        f"Esperava exatamente 1 CSV de crosswalk em {DATA_DIR} (nome contendo 'crosswalk'), "
        f"encontrei {len(_crosswalk_candidates)}: {_crosswalk_candidates}. Confira se voce "
        f"fez upload de crosswalk_manual_recebido.csv junto com os 3 CSVs na secao 2."
    )
CROSSWALK_PATH = _crosswalk_candidates[0]

# Confere se o arquivo e mesmo um CSV de texto antes de tentar ler (evita erro criptico
# quando o arquivo enviado e, por exemplo, uma planilha do Numbers/Excel salva com
# extensao .csv por engano -- esses formatos comecam com a assinatura de um arquivo zip).
with open(CROSSWALK_PATH, "rb") as _f:
    _head = _f.read(4)
if _head[:2] == b"PK":
    raise ValueError(
        f"{os.path.basename(CROSSWALK_PATH)} nao e um CSV de texto -- o conteudo comeca "
        f"com a assinatura de um arquivo zip, sinal de que e uma planilha do Numbers/Excel/"
        f"Google Sheets salva (ou so renomeada) com extensao .csv, nao exportada de fato "
        f"como CSV. Reabra a planilha original e use 'Exportar Para > CSV' (Numbers) ou "
        f"'Fazer download > Valores separados por virgula (.csv)' (Google Sheets) ou "
        f"'Salvar Como > CSV UTF-8' (Excel), depois envie de novo o arquivo gerado."
    )

# O CSV pode ter sido exportado com separador "," ou ";" (Excel/Numbers em configuracao
# regional PT-BR costumam usar ";" para nao colidir com a virgula decimal) e em encoding
# UTF-8 ou latin-1/cp1252. Em vez de assumir um formato fixo e travar com um erro pouco
# informativo, tenta as combinacoes mais comuns e confirma pelo nome das colunas esperado.
_EXPECTED_COLS = {"codigo", "presente_em", "rotulo_2023", "coluna_original_2023",
                   "rotulo_2024", "coluna_original_2024", "rotulo_2025",
                   "coluna_original_2025", "coluna_unificada_final", "ok_ou_ajustar"}

crosswalk = None
for _enc in ("utf-8", "latin-1"):
    for _sep in (",", ";", "	"):
        try:
            _df = pd.read_csv(CROSSWALK_PATH, dtype=str, keep_default_na=False, encoding=_enc, sep=_sep)
        except Exception:
            continue
        if _EXPECTED_COLS.issubset(set(_df.columns)):
            crosswalk = _df
            if _sep != "," or _enc != "utf-8":
                print(f"aviso: CSV lido com separador {_sep!r} / encoding {_enc!r} "
                      f"(diferente do padrao ',' / 'utf-8').")
            break
    if crosswalk is not None:
        break

if crosswalk is None:
    with open(CROSSWALK_PATH, "r", encoding="utf-8", errors="replace") as _f:
        _primeiras_linhas = [next(_f, "") for _ in range(5)]
    raise ValueError(
        f"Nao consegui ler {os.path.basename(CROSSWALK_PATH)} como a planilha de crosswalk "
        f"esperada (tentei separador ',' / ';' / tab, em UTF-8 e latin-1, e nenhuma combinacao "
        f"produziu as colunas esperadas: {sorted(_EXPECTED_COLS)}).\n"
        f"Primeiras linhas do arquivo, para diagnostico:\n" + "".join(_primeiras_linhas) +
        f"\nReexporte a planilha original como CSV puro (Numbers: Exportar Para > CSV; "
        f"Google Sheets: Fazer download > Valores separados por virgula; Excel: Salvar Como > "
        f"CSV UTF-8) e envie de novo, ou use direto o crosswalk_manual_recebido.csv ja pronto."
    )

print(f"crosswalk carregado: {os.path.basename(CROSSWALK_PATH)} ({len(crosswalk)} linhas)")

# Validacao de cobertura: toda coluna original dos 3 CSVs enviados precisa aparecer em
# exatamente uma linha do crosswalk.
_real_cols = {2023: set(df23_raw.columns), 2024: set(df24_raw.columns), 2025: set(df25_raw.columns)}
_problemas = []
for year in (2023, 2024, 2025):
    field = f"coluna_original_{year}"
    used = [c for c in crosswalk[field].tolist() if c]
    dupes = pd.Series(used).value_counts()
    dupes = dupes[dupes > 1]
    missing = _real_cols[year] - set(used)
    extra = set(used) - _real_cols[year]
    if len(dupes) or missing or extra:
        _problemas.append((year, dupes, missing, extra))
if _problemas:
    for year, dupes, missing, extra in _problemas:
        print(f"!!! problema no ano {year}: duplicadas={len(dupes)} faltando={len(missing)} sobrando={len(extra)}")
        if len(dupes):
            print(dupes)
        if missing:
            print("faltando:", missing)
        if extra:
            print("sobrando:", extra)
    raise ValueError("O crosswalk enviado nao bate 100% com as colunas dos CSVs enviados -- ver problemas acima.")
print("cobertura de colunas originais: OK nos 3 anos (nenhuma perdida, duplicada ou invalida)")

In [ ]:
# Monta o mapeamento final (codigo -> {ano: {orig, label}}) a partir da decisao manual de
# cada linha do crosswalk: OK/AJUSTAR (anos ficam juntos) ou SEPARAR:20AA (esse ano sai para
# coluna propria).
import re
import unicodedata


def slugify(text, max_len=40):
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text[:max_len].rstrip("_")


def unified_col_name(code_, label):
    code_part = "q" + code_.replace(".", "_")
    label_part = slugify(label)
    return f"{code_part}__{label_part}" if label_part else code_part


def canonical_label(entry):
    for year in (2025, 2024, 2023):
        if year in entry:
            return entry[year]["label"]
    return ""


mapping = {}
n_separar = 0
for _, row in crosswalk.iterrows():
    code_ = row["codigo"]
    status = row["ok_ou_ajustar"].strip().upper()
    separar_anos = set()
    if status.startswith("SEPARAR"):
        for part in status.split(";"):
            part = part.strip()
            if ":" in part:
                try:
                    separar_anos.add(int(part.split(":", 1)[1]))
                except ValueError:
                    pass

    entry_principal = {}
    for year in (2023, 2024, 2025):
        orig = row[f"coluna_original_{year}"]
        label = row[f"rotulo_{year}"]
        if not orig:
            continue
        if year in separar_anos:
            sep_code = f"{code_}__var{year}"
            mapping[sep_code] = {year: {"orig": orig, "label": label}}
            n_separar += 1
        else:
            entry_principal[year] = {"orig": orig, "label": label}

    if entry_principal:
        mapping[code_] = entry_principal

print(f"codigos no mapping final: {len(mapping)}  (colunas separadas manualmente: {n_separar})")


def build_rename_maps(mapping):
    renames = {2023: {}, 2024: {}, 2025: {}}
    data_dict_rows = []
    used_names = set()
    for code_, entry in mapping.items():
        label = canonical_label(entry)
        base_name = unified_col_name(code_, label)
        name = base_name
        i = 2
        while name in used_names:
            name = f"{base_name}_{i}"
            i += 1
        used_names.add(name)
        for year in (2023, 2024, 2025):
            if year in entry:
                renames[year][entry[year]["orig"]] = name
        data_dict_rows.append({
            "codigo_pergunta": code_,
            "coluna_unificada": name,
            "presente_em": ",".join(str(y) for y in sorted(entry.keys())),
            "rotulo_2023": entry.get(2023, {}).get("label", ""),
            "rotulo_2024": entry.get(2024, {}).get("label", ""),
            "rotulo_2025": entry.get(2025, {}).get("label", ""),
        })
    return renames, data_dict_rows


RENAMES, DATA_DICT_ROWS = build_rename_maps(mapping)

n_all3 = sum(1 for v in mapping.values() if len(v) == 3)
n_2 = sum(1 for v in mapping.values() if len(v) == 2)
n_1 = sum(1 for v in mapping.values() if len(v) == 1)
print(f"codigos de pergunta unificados: {len(mapping)}")
print(f"presentes nos 3 anos: {n_all3} | em 2 anos: {n_2} | exclusivos de 1 ano: {n_1}")

## 5. Decisão da estratégia de consolidação

**UNION (empilhamento de linhas), não JOIN.**

As 3 bases representam o mesmo tipo de registro — uma resposta de pesquisa — coletado em anos
diferentes, de pessoas diferentes. Não existe (nem faria sentido inventar) uma chave para
relacionar uma linha de 2023 a uma linha de 2024/2025 linha a linha. A forma correta de
consolidar é empilhar as respostas em uma única tabela longitudinal, com uma coluna
`ano_pesquisa` marcando a origem de cada linha.

**A limpeza acontece antes do union, não depois**: cada base é tratada e tem suas colunas
renomeadas para o padrão unificado *antes* de serem unidas. Se a união fosse feita com os nomes
de coluna originais, o Spark trataria colunas como `P1_a` (2023) e `1.a_idade` (2024) como
colunas completamente diferentes — a base final ficaria fragmentada em vez de consolidada.

## 6. Tratamento (limpeza + padronização) de cada base

Antes de aplicar as transformações em PySpark, identificamos rapidamente (com uma leitura leve,
pandas, apenas para fins de perfilamento de tipos — a transformação pesada continua sendo feita
em PySpark) quais colunas são:

- **numéricas** (idade)
- **timestamp** (data/hora de envio, só em 2024/2025)
- **flags de múltipla escolha / binárias** (0/1, ou "True"/"False" dependendo do ano)

Isso evita ter que decidir o tipo de cada uma das ~490 colunas manualmente.

In [ ]:
_pd = {y: pd.read_csv(p, low_memory=False) for y, p in FILES.items()}

FLAG_COLS = set()
INT_COLS = set()
TS_COLS = set()

for year, df_pd in _pd.items():
    for orig_col, unified in RENAMES[year].items():
        if orig_col not in df_pd.columns:
            continue
        s = df_pd[orig_col]
        if s.dtype.kind == 'f':
            non_null = s.dropna().unique()
            if len(non_null) > 0 and set(non_null.tolist()).issubset({0.0, 1.0}):
                FLAG_COLS.add(unified)

for code_, entry in mapping.items():
    if code_ == "1.a":  # idade
        for year in entry:
            INT_COLS.add(RENAMES[year][entry[year]["orig"]])
    if code_ == "0.d":  # timestamp de envio
        for year in entry:
            TS_COLS.add(RENAMES[year][entry[year]["orig"]])

del _pd  # libera memoria, o pandas so foi usado para o perfilamento de tipos

print("colunas flag binaria (0/1):", len(FLAG_COLS))
print("colunas numericas (idade):", INT_COLS)
print("colunas timestamp:", TS_COLS)

In [ ]:
# Uma mesma pergunta binaria aparece com representacoes diferentes conforme o ano:
# 0.0/1.0 (numerico) ou "TRUE"/"FALSE" (texto, ex.: "atua_como_gestor", "possui_data_lake").
# Padronizamos tudo para inteiro 0/1, mantendo nulo quando nao respondida/nao aplicavel.
_TRUE_TOKENS = ("1", "1.0", "TRUE", "T", "VERDADEIRO", "SIM", "YES")
_FALSE_TOKENS = ("0", "0.0", "FALSE", "F", "FALSO", "NAO", "NÃO", "NO")


def build_column_expr(new_name):
    """Para UMA coluna ja renomeada: trim (padronizacao de texto) + cast de tipo.
    Tudo em uma unica expressao Column, para evitar encadear centenas de withColumn
    (o que gera um plano de analise profundo demais e pode estourar a stack da JVM)."""
    c = F.trim(F.col(new_name))
    if new_name in INT_COLS:
        c = F.expr(f"try_cast(`{new_name}` as double)").cast("int")
    elif new_name in FLAG_COLS:
        upper_c = F.upper(c)
        c = (F.when(upper_c.isin(*_TRUE_TOKENS), F.lit(1))
              .when(upper_c.isin(*_FALSE_TOKENS), F.lit(0))
              .otherwise(F.lit(None).cast("int")))
    elif new_name in TS_COLS:
        c = F.to_timestamp(c, "dd/MM/yyyy HH:mm:ss")
    return c.alias(new_name)


def load_and_treat(year):
    path = FILES[year]
    rename_map = RENAMES[year]

    raw = spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
    n_raw = raw.count()

    # Renomeia por POSICAO (toDF): os cabecalhos originais de 2023 contem pontos dentro
    # do proprio texto do rotulo, o que confundiria o parser de nomes do Spark
    # (F.col/df[nome]) mesmo com o nome entre crases.
    new_names = [rename_map.get(orig, orig) for orig in raw.columns]
    renamed = raw.toDF(*new_names)

    # Trim + casts de tipo em um unico select()
    df = renamed.select(*[build_column_expr(c) for c in renamed.columns])

    # Duplicatas exatas dentro do proprio ano
    n_before = df.count()
    df = df.dropDuplicates()
    n_after = df.count()

    # Proveniencia
    df = df.select(
        F.lit(year).alias("ano_pesquisa"),
        F.lit(os.path.basename(path)).alias("arquivo_origem"),
        *[F.col(c) for c in df.columns],
    )

    print(f"[{year}] linhas brutas: {n_raw} | duplicatas removidas: {n_before - n_after} | linhas finais: {n_after}")
    return df, n_after


df23, n23 = load_and_treat(2023)
df24, n24 = load_and_treat(2024)
df25, n25 = load_and_treat(2025)

## 7. Consolidação final e validação

In [ ]:
# UNION por nome de coluna; colunas ausentes num ano viram null automaticamente
consolidado = df23.unionByName(df24, allowMissingColumns=True) \
                   .unionByName(df25, allowMissingColumns=True)

outras_cols = [c for c in consolidado.columns if c not in ("ano_pesquisa", "arquivo_origem")]
consolidado = consolidado.select("ano_pesquisa", "arquivo_origem", *outras_cols)

total_esperado = n23 + n24 + n25
total_real = consolidado.count()
print(f"linhas esperadas: {total_esperado} | linhas na base consolidada: {total_real}")
assert total_esperado == total_real, "contagem divergente apos o union!"

print("colunas na base final:", len(consolidado.columns))
consolidado.groupBy("ano_pesquisa").count().orderBy("ano_pesquisa").show()

n_dedup_final = consolidado.dropDuplicates().count()
print("linhas 100% identicas na base final:", total_real - n_dedup_final)

In [ ]:
consolidado.printSchema()

In [ ]:
consolidado.show(5, truncate=30)

## 8. Exportação da base final e relatório

O parquet e gravado **particionado por `ano_pesquisa`** — é a dimensão que qualquer
consumo posterior (camada gold, dashboard, notebook de análise) vai filtrar primeiro. Com a
partição, um `.filter(F.col("ano_pesquisa") == 2025)` lê só os arquivos daquele ano, em vez de
escanear a base inteira (*partition pruning*). O Spark remove a coluna de partição do conteúdo dos
arquivos e a reconstrói a partir do caminho ao ler de volta — não muda o schema lógico da tabela, e
`spark.read.parquet(...)` continua devolvendo `ano_pesquisa` como coluna normal.

In [ ]:
OUT_DIR = "/content/output"
os.makedirs(OUT_DIR, exist_ok=True)

(consolidado.write.mode("overwrite")
 .partitionBy("ano_pesquisa")
 .parquet(f"{OUT_DIR}/state_of_data_br_consolidado_parquet"))
(consolidado.coalesce(1).write.mode("overwrite").option("header", True)
 .csv(f"{OUT_DIR}/state_of_data_br_consolidado_csv"))

import csv as _csv
with open(f"{OUT_DIR}/dicionario_de_dados.csv", "w", newline="", encoding="utf-8") as f:
    w = _csv.DictWriter(f, fieldnames=["codigo_pergunta", "coluna_unificada", "presente_em",
                                        "rotulo_2023", "rotulo_2024", "rotulo_2025"])
    w.writeheader()
    for row in DATA_DICT_ROWS:
        w.writerow(row)

print("arquivos gravados em", OUT_DIR)
print(os.listdir(OUT_DIR))

In [ ]:
# Baixar os resultados (opcional, se estiver rodando no Colab)
try:
    import shutil
    from google.colab import files
    shutil.make_archive("/content/state_of_data_br_resultado", "zip", OUT_DIR)
    files.download("/content/state_of_data_br_resultado.zip")
except ImportError:
    print("Fora do Colab - arquivos disponiveis em", OUT_DIR)

### Resumo final

- **Estratégia de consolidação**: UNION (empilhamento), pois as 3 bases são edições anuais da
  mesma pesquisa aplicadas a respondentes diferentes — sem chave em comum para join relacional.
- **Tratamento aplicado antes da consolidação**: renomeação de colunas via mapeamento por código
  de pergunta, trim de texto, padronização de tipos (idade, timestamp, flags binárias com
  representações inconsistentes entre anos) e remoção de duplicatas exatas por base.
- **Resultado**: uma única base longitudinal com uma linha por resposta de pesquisa, coluna
  `ano_pesquisa` preservando a origem, e um dicionário de dados documentando a correspondência
  de cada coluna final com as perguntas originais de cada edição.